In [71]:
import psycopg2

def get_conn():
    return psycopg2.connect(
        host="localhost",
        port=5432,
        dbname="postgres",
        user="postgres",
        password="",
    )

In [72]:
from pathlib import Path
from datetime import date, timedelta, datetime
import os
import re

import numpy as np
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from concurrent.futures import ThreadPoolExecutor, as_completed

# =========================
# 고정 경로 (요구사항)
# =========================
BASE_DIR = Path(r"C:\Users\user\Desktop\machinlog\FCT")
START_DATE = date(2025, 10, 1)
END_DATE   = date.today()

# 매핑 CSV (첨부파일을 C:\data에 둔 기준)
MAP_PD  = Path(r"C:\data\PD.csv")
MAP_NPD = Path(r"C:\data\Non-PD.csv")

# =========================
# DB 설정 (요구사항)
# =========================
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

# 타겟 스키마/테이블 (요구사항)
TGT_SCHEMA = "c1_fct_detail"
TGT_TABLE  = "fct_details"

# 소스(매칭) 스키마/테이블
SRC_SCHEMA = "a2_fct_table"
SRC_TABLE  = "fct_table"

# 성능/인코딩
ANSI_LOG_ENCODING = "cp949"
MAX_WORKERS = min(16, (os.cpu_count() or 4) * 2)

# 결과 라인 패턴
OK_PAT = re.compile(r"테스트\s*결과\s*:\s*OK")
NG_PAT = re.compile(r"테스트\s*결과\s*:\s*NG")

In [73]:
def read_csv_flex(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding="cp949")
    except Exception:
        return pd.read_csv(path, encoding="utf-8-sig")

def normalize_mapping(df: pd.DataFrame, remark_value: str) -> pd.DataFrame:
    df = df.copy()

    # seq 컬럼이 없으면 행 순서(1..N)를 seq로 사용
    if "seq" not in df.columns:
        df.insert(0, "seq", np.arange(1, len(df) + 1, dtype=int))

    need_cols = ["seq", "problem1", "problem2", "problem3", "problem4", "test_item"]
    for c in need_cols:
        if c not in df.columns:
            df[c] = None

    df["seq"] = pd.to_numeric(df["seq"], errors="coerce").astype("Int64")
    df["remark"] = remark_value
    return df[need_cols + ["remark"]]

map_pd  = normalize_mapping(read_csv_flex(MAP_PD),  "PD")
map_npd = normalize_mapping(read_csv_flex(MAP_NPD), "Non-PD")
MAP_DF  = pd.concat([map_pd, map_npd], ignore_index=True)

print("[INFO] MAP_DF:", MAP_DF.shape)
display(MAP_DF.head(5))

[INFO] MAP_DF: (144, 7)


,seq,problem1,problem2,problem3,problem4,test_item,remark
0,1,dmm,NaN,NaN,NaN,1.00_dmm_c_rng_set,PD
1,2,relay_board,NaN,NaN,NaN,1.00_d_sig_val_090_set,PD
2,3,load_c,NaN,NaN,NaN,1.00_load_c_cc_rng_set,PD
3,4,board,NaN,NaN,NaN,1.00_d_sig_val_000_set,PD
4,5,dmm,NaN,NaN,NaN,1.00_dmm_dc_v_set,PD


In [74]:
def iter_dates(start, end):
    d = start
    while d <= end:
        yield d
        d += timedelta(days=1)

def detect_remark_from_folder(folder_name: str):
    u = folder_name.upper()
    if "PD NONE" in u:
        return "Non-PD"
    if "PD" in u:
        return "PD"
    return None

def list_candidate_files_strict():
    """
    대상: BASE/YYYY/MM/DD/{target_folder}/{files}
    - DD 바로 아래 폴더(target_folder) 안의 파일만
    """
    candidates = []  # (Path, remark)
    for d in iter_dates(START_DATE, END_DATE):
        day_dir = BASE_DIR / f"{d.year:04d}" / f"{d.month:02d}" / f"{d.day:02d}"
        if not day_dir.exists():
            continue

        for target_folder in day_dir.iterdir():
            if not target_folder.is_dir():
                continue

            remark = detect_remark_from_folder(target_folder.name)

            for fp in target_folder.iterdir():  # 1-depth만 (속도 최적)
                if fp.is_file() and "_" in fp.name:
                    candidates.append((fp, remark))

    return candidates

cands = list_candidate_files_strict()
print(f"[INFO] candidates: {len(cands):,}")
print("[INFO] sample:", cands[:5])

[INFO] candidates: 293
[INFO] sample: [(WindowsPath('C:/Users/user/Desktop/machinlog/FCT/2025/10/10/FORD A+C PD FULL 35915729/BA1WJ24068500081SJ8T-14F014-AC_20251010_081513.txt'), 'PD'), (WindowsPath('C:/Users/user/Desktop/machinlog/FCT/2025/10/10/FORD A+C PD FULL 35915729/BA1WJ24068500081SJ8T-14F014-AC_20251010_082225.txt'), 'PD'), (WindowsPath('C:/Users/user/Desktop/machinlog/FCT/2025/10/10/FORD A+C PD FULL 35915729/BA1WJ24068500081SJ8T-14F014-AC_20251010_202343.txt'), 'PD'), (WindowsPath('C:/Users/user/Desktop/machinlog/FCT/2025/10/10/FORD A+C PD FULL 35915729/BA1WJ24068500086SJ8T-14F014-AC_20251010_081507.txt'), 'PD'), (WindowsPath('C:/Users/user/Desktop/machinlog/FCT/2025/10/10/FORD A+C PD FULL 35915729/BA1WJ24068500086SJ8T-14F014-AC_20251010_081834.txt'), 'PD')]


In [75]:
def t_to_cs(tstr: str) -> int:
    hh = int(tstr[0:2]); mm = int(tstr[3:5]); ss = int(tstr[6:8]); cs = int(tstr[9:11])
    return ((hh*60 + mm)*60 + ss) * 100 + cs

def parse_one_file_fast(fp: Path, remark: str):
    name = fp.name
    barcode_information = name.split("_", 1)[0] if "_" in name else name

    rows = []
    try:
        with open(fp, "r", encoding=ANSI_LOG_ENCODING, errors="ignore") as f:
            for line in f:
                line = line.strip()
                if not line.startswith("[") or len(line) < 13:
                    continue
                tstr = line[1:12]  # HH:MM:SS.xx
                if len(tstr) != 11 or tstr[2] != ":" or tstr[5] != ":" or tstr[8] != ".":
                    continue
                contents = line[line.find("]")+1:].strip()
                rows.append((t_to_cs(tstr), tstr, contents))
    except Exception:
        return None

    if not rows:
        return None

    rows.sort(key=lambda x: x[0])

    out = []
    prev_cs = None
    seq = 0

    for cs, tstr, contents in rows:
        ct = None
        if prev_cs is not None:
            ct = round((cs - prev_cs) / 100.0, 2)
        prev_cs = cs

        result = None
        is_ok = bool(OK_PAT.search(contents))
        is_ng = bool(NG_PAT.search(contents))
        if is_ok or is_ng:
            seq += 1
            result = "PASS" if is_ok else "FAIL"
            seq_val = seq
        else:
            seq_val = pd.NA

        out.append({
            "barcode_information": barcode_information,
            "remark": remark,
            "contents": contents,
            "time": tstr,
            "ct": ct,
            "result": result,
            "seq": seq_val,
            "file_path": str(fp),
        })

    return out

def build_df_from_files(cands, MAP_DF):
    parsed = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(parse_one_file_fast, fp, remark): fp for fp, remark in cands}
        for fut in as_completed(futs):
            r = fut.result()
            if r:
                parsed.extend(r)

    df = pd.DataFrame(parsed)
    if df.empty:
        return df

    df["seq"] = df["seq"].astype("Int64")
    df = df.merge(MAP_DF, how="left", on=["remark", "seq"])
    df.drop(columns=["seq"], inplace=True)

    # processed_at
    df["processed_at"] = datetime.now()

    # group: barcode_information별 1..N (DB 스키마 요구 충족용)
    uniq = pd.unique(df["barcode_information"])
    gmap = {bc: i+1 for i, bc in enumerate(uniq)}
    df.insert(0, "group", df["barcode_information"].map(gmap).astype("Int64"))

    # id는 DB에서 bigserial로 생성하므로 DF에는 없음(Insert 시 제외)
    return df

df_parsed = build_df_from_files(cands, MAP_DF)
print("[INFO] parsed df:", df_parsed.shape)
display(df_parsed.head(10))

[INFO] parsed df: (54394, 14)


,group,barcode_information,remark,contents,time,ct,result,file_path,problem1,problem2,problem3,problem4,test_item,processed_at
0,1,BA1WJ24068500088SJ8T-14F014-AC,PD,MES 이전공정 체크 SKIP,20:23:54.52,NaN,None,C:\Users\user\Desktop\machinlog\FCT\2025\10\10...,NaN,NaN,NaN,NaN,NaN,2025-12-15 18:35:24.633246
1,1,BA1WJ24068500088SJ8T-14F014-AC,PD,START :: DMM SET CURRENT RANGE,20:23:56.70,2.18,None,C:\Users\user\Desktop\machinlog\FCT\2025\10\10...,NaN,NaN,NaN,NaN,NaN,2025-12-15 18:35:24.633246
2,1,BA1WJ24068500088SJ8T-14F014-AC,PD,MODE SET :: CURR_RANG,20:23:56.76,0.06,None,C:\Users\user\Desktop\machinlog\FCT\2025\10\10...,NaN,NaN,NaN,NaN,NaN,2025-12-15 18:35:24.633246
3,1,BA1WJ24068500088SJ8T-14F014-AC,PD,테스트 결과 : OK,20:23:56.95,0.19,PASS,C:\Users\user\Desktop\machinlog\FCT\2025\10\10...,dmm,NaN,NaN,NaN,1.00_dmm_c_rng_set,2025-12-15 18:35:24.633246
4,1,BA1WJ24068500088SJ8T-14F014-AC,PD,START :: DO SET,20:23:56.97,0.02,None,C:\Users\user\Desktop\machinlog\FCT\2025\10\10...,NaN,NaN,NaN,NaN,NaN,2025-12-15 18:35:24.633246
5,1,BA1WJ24068500088SJ8T-14F014-AC,PD,DO_SET Value :: 0900800080005000,20:23:56.99,0.02,None,C:\Users\user\Desktop\machinlog\FCT\2025\10\10...,NaN,NaN,NaN,NaN,NaN,2025-12-15 18:35:24.633246
6,1,BA1WJ24068500088SJ8T-14F014-AC,PD,테스트 결과 : OK,20:23:57.01,0.02,PASS,C:\Users\user\Desktop\machinlog\FCT\2025\10\10...,relay_board,NaN,NaN,NaN,1.00_d_sig_val_090_set,2025-12-15 18:35:24.633246
7,1,BA1WJ24068500088SJ8T-14F014-AC,PD,START :: LOAD_C SET CC MODE,20:23:57.03,0.02,None,C:\Users\user\Desktop\machinlog\FCT\2025\10\10...,NaN,NaN,NaN,NaN,NaN,2025-12-15 18:35:24.633246
8,1,BA1WJ24068500088SJ8T-14F014-AC,PD,Return Value :: OK,20:23:57.06,0.03,None,C:\Users\user\Desktop\machinlog\FCT\2025\10\10...,NaN,NaN,NaN,NaN,NaN,2025-12-15 18:35:24.633246
9,1,BA1WJ24068500088SJ8T-14F014-AC,PD,테스트 결과 : OK,20:23:57.07,0.01,PASS,C:\Users\user\Desktop\machinlog\FCT\2025\10\10...,load_c,NaN,NaN,NaN,1.00_load_c_cc_rng_set,2025-12-15 18:35:24.633246


In [76]:
inserted_rows = insert_df(df_ready)
print("inserted_rows =", inserted_rows)

[INFO] insert_df() started
[INFO] ensuring target schema/table...
[INFO] ALTER skipped: all columns exist
[INFO] incoming files detected: 293
[INFO] checking existing file_path in DB (293)
[INFO] existing file_path fetched: 0
[INFO] existing files: 0
[INFO] new files: 293
[INFO] rows to insert: 54,394
[INFO] inserting into c1_fct_detail.fct_details ...
[INFO] insert completed: 54,394 rows (5.84s)
inserted_rows = 54394
